# 01 - WTO Dataset Creation

Extracts structured fields from one-page WTO dispute-summary PDFs and creates the initial case-law dataset.

**Expected input:** PDF files in `data/raw/wto_case_pdfs/`  
**Generated output:** `data/processed/wto_cases.csv`

> Install dependencies once from the repository root with `pip install -r requirements.txt`. Run the notebooks in numerical order.

In [ ]:
from pathlib import Path

# Resolve repository paths whether Jupyter starts in the repository root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw" / "wto_case_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = PROCESSED_DIR / "wto_cases.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "preprocessed_wto_cases.csv"

print(f"Project root: {PROJECT_ROOT}")


In [2]:
# Step Two: Import the required packages 
import pdfplumber
import pandas as pd
import re
import os

# Step Three: Define the Path to the folder containing WTO case PDFs
folder_path = RAW_PDF_DIR

# Step Four: Function to extract case details from PDF files
def extract_case_details(pdf_path, file_name):
    case_details = {
        "Case ID": None,  # Extracted from filename
        "Title": None,
        "Complainant": None,  
        "Respondent": None,   
        "Date": None,         
        "Articles": None,     
        "Summary": None       
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            first_page = pdf.pages[0] if pdf.pages else None
            first_page_text = first_page.extract_text() if first_page else ""

            # Extract Case ID from filename (remove "sum_e" and ".pdf")
            cleaned_case_id = file_name.replace("sum_e", "").replace(".pdf", "").strip()
            case_details["Case ID"] = cleaned_case_id

            # Extract Title
            match = re.search(r"(?i)(.*?)\s*\(?DS(\d{1,4})\)?", first_page_text)
            if match:
                case_details["Title"] = match.group(1).strip()

            # Extract Complainant and Respondent
            match = re.search(r"Complainant\s+([\s\S]+?)\nRespondent", first_page_text)
            if match:
                case_details["Complainant"] = clean_country_name(match.group(1))

            match = re.search(r"Respondent\s+([\s\S]+?)\n", first_page_text)
            if match:
                case_details["Respondent"] = clean_country_name(match.group(1))

            # Extract Date (from "Adoption" field)
            match = re.search(r"Adoption\s*(\d{1,2}\s+\w+\s+\d{4})", first_page_text)
            if match:
                case_details["Date"] = match.group(1)

            # Extract "AGREEMENT" column from the blue table for Articles 
            if first_page:
                tables = first_page.extract_table()
                if tables:
                    header_row = tables[0]  # First row is the column names
                    if "AGREEMENT" in header_row:
                        agreement_index = header_row.index("AGREEMENT")

                        # Extract the corresponding value from the next row
                        for row in tables[1:]:  # Skip header
                            if len(row) > agreement_index:  # Ensure valid index
                                case_details["Articles"] = row[agreement_index].strip()
                                break  # Take the first valid occurrence
            
            # Extract Summary (from "2. SUMMARY OF KEY PANEL" section)
            match = re.search(r"2\.\s*SUMMARY OF KEY PANEL(.*?)(?:\n[A-Z\s]+:|\n[A-Z\s]+\n|$)", first_page_text, re.DOTALL)
            if match:
                case_details["Summary"] = match.group(1).strip()

    except Exception as e:
        print(f"❌ Error processing {pdf_path}: {e}")

    return case_details

    
# Step Five: Additional extraction functions 
# Function to clean extracted country names (remove extra text)
def clean_country_name(raw_text):
    if raw_text:
        country = raw_text.split("\n")[0]  # Take only the first line (country name)
        country = re.split(r",|;|Circulation|Arts?", country)[0]  # Remove unnecessary text
        return country.strip()
    return None


# List to store extracted data
all_cases = []

# Loop through all PDFs in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".pdf"):
        pdf_path = os.path.join(folder_path, file_name)
        
        # Extract details, passing the filename for Case ID extraction
        case_data = extract_case_details(pdf_path, file_name)
        all_cases.append(case_data)

# Step Six: Convert to dataFrame
df = pd.DataFrame(all_cases)

# Step Seven: Save to CSV
csv_path = RAW_DATA_PATH
df.to_csv(csv_path, index=False)

print(f"✅ CSV file saved at: {csv_path}")


✅ CSV file saved at: data/processed/wto_cases.csv
